In [ ]:
import os
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.preprocessing import LabelEncoder

# 1. Dosya Yolunu Otomatik Bulma
base_path = ""
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == 'train.csv':
            base_path = dirname + "/"
            break

print(f"Bulunan klasör yolu: {base_path}")

# 2. Ham Verileri Okuma
print("Ham veriler yükleniyor...")
train = pd.read_csv(base_path + "train.csv")
stores = pd.read_csv(base_path + "stores.csv")
oil = pd.read_csv(base_path + "oil.csv")
holidays = pd.read_csv(base_path + "holidays_events.csv")
test = pd.read_csv(base_path + "test.csv")

# Tarih formatları
train['date'] = pd.to_datetime(train['date'])
oil['date'] = pd.to_datetime(oil['date'])
holidays['date'] = pd.to_datetime(holidays['date'])
test['date'] = pd.to_datetime(test['date'])

holidays = holidays.drop_duplicates(subset=['date'], keep='first')

# 3. Verileri Birleştirme (Merge)
print("Veriler birleştiriliyor...")
df = train.merge(stores, on='store_nbr', how='left')
df = df.merge(oil, on='date', how='left')
df = df.merge(holidays, on='date', how='left')
df['dcoilwtico'] = df['dcoilwtico'].ffill()

df_test = test.merge(stores, on='store_nbr', how='left')
df_test = df_test.merge(oil, on='date', how='left')
df_test = df_test.merge(holidays, on='date', how='left')
df_test['dcoilwtico'] = df_test['dcoilwtico'].ffill()

for data in [df, df_test]:
    data.rename(columns={"type_x": "store_type", "type_y": "holiday_type"}, inplace=True, errors='ignore')
    for col in ['holiday_type', 'locale', 'locale_name']:
        data[col] = data[col].fillna('Normal')

# Tarih Özellikleri (Feature Engineering)
for data in [df, df_test]:
    data['year'] = data['date'].dt.year
    data['month'] = data['date'].dt.month
    data['day'] = data['date'].dt.day
    data['day_of_week'] = data['date'].dt.dayofweek
    data['is_weekend'] = data['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
    data['is_payday'] = ((data['date'].dt.is_month_end) | (data['day'] == 15)).astype(int)

# 4. Kategorik Kodlama (Label Encoding)
print("Kategorik veriler sayısallaştırılıyor...")
categorical_cols = ['family', 'city', 'state', 'store_type', 'cluster', 'holiday_type', 'locale', 'locale_name']

for col in categorical_cols:
    le = LabelEncoder()
    le.fit(pd.concat([df[col].astype(str), df_test[col].astype(str)]))
    df[col] = le.transform(df[col].astype(str))
    df_test[col] = le.transform(df_test[col].astype(str))

df = df.drop(['description', 'transferred'], axis=1, errors='ignore')
df_test = df_test.drop(['description', 'transferred'], axis=1, errors='ignore')

# 5. Geçmiş Satış Ortalamaları (Target Encoding)
print("Geçmiş satış ortalamaları hesaplanıyor...")
store_family_sales = df.groupby(['store_nbr', 'family'])['sales'].mean().reset_index()
store_family_sales.rename(columns={'sales': 'avg_sales_per_store_family'}, inplace=True)

df = df.merge(store_family_sales, on=['store_nbr', 'family'], how='left')
df_test = df_test.merge(store_family_sales, on=['store_nbr', 'family'], how='left')
df_test['avg_sales_per_store_family'] = df_test['avg_sales_per_store_family'].fillna(0)

# 6. Model Eğitimi (Logaritmik Dönüşüm ile)
print("XGBoost Modeli eğitiliyor...")
features = [c for c in df.columns if c not in ['id', 'date', 'sales']]

model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
model.fit(df[features], np.log1p(df['sales']))
print("Model eğitimi tamamlandı!")

# 7. Tahmin ve Dosya Üretimi
print("Test seti tahminleri yapılıyor...")
predictions = np.expm1(model.predict(df_test[features]))
predictions = [max(0, p) for p in predictions]

submission = pd.DataFrame({
    'id': df_test['id'],
    'sales': predictions
})
submission.to_csv("submission.csv", index=False)
print("submission.csv başarıyla oluşturuldu!")